# MCP Security Observability and SOC Integration

Build an evidence-bound path from an authenticated MCP event to a candidate alert, an idempotent SOC case, an authorized containment action, and an independently verified outcome. The notebook is offline and reuses `lab.py`; it creates no external SIEM or control-plane resources.

## Outcomes

You will inspect event authenticity, producer scope, privacy minimization, OpenTelemetry and OCSF projections, replay behavior, event-time detection, evidence provenance, case routing, response approval, outcome verification, and labelled quality metrics.

## Authority boundary

A model, log, detector, or alert proposes an interpretation. Trusted application code validates evidence, resolves tenant and identity, authorizes effects, persists state, and verifies outcomes. Keep this boundary visible through every cell.

In [ ]:
import runpy
from dataclasses import replace
from datetime import timedelta
module = runpy.run_path('lab.py')
sorted(name for name in ('EventEmitter', 'EventNormalizer', 'DetectionEngine', 'CaseRouter', 'ApprovalService') if name in module)

## Scenario

A multi-tenant support platform exposes reviewed `ticket.read`, `ticket.update`, and `ticket.export` MCP actions. Its approved server may reach only `tickets.example.com`. A signed gateway event reports a successful export to an unapproved destination.

In [ ]:
emitter, normalizer, store, detector, router, approvals, controller = module['demo_components']()
(emitter.emitter_id, list(normalizer.inventory.records), router.allowed_routes)

## 1. Sign the exact source event

The Ed25519 signature authenticates the exact schema, emitter, and event bytes. Separate producer bindings decide which tenants, workloads, and event kinds that key may attest. A signature alone is not authorization or accuracy.

In [ ]:
NOW = module['NOW']
signed = module['event_fixture'](
    emitter, event_id='EV-EGRESS-NB', kind=module['EventKind'].NETWORK,
    action='ticket.export', destination='exfil.example',
    outcome=module['Outcome'].SUCCEEDED,
)
(signed.schema_version, signed.emitter_id, len(signed.signature), signed.event.operation_id)

## 2. Authenticate, minimize, and enrich

The normalizer verifies signature, identifiers, trace context, timestamps, producer scope, privacy constraints, and trusted inventory before it exposes a typed event to detection. Ownership and SOC routing come from inventory—not the source event.

In [ ]:
event = normalizer.normalize(signed, NOW + timedelta(seconds=2))
{
    'event_id': event.event_id, 'tenant': event.tenant_id,
    'owner': event.owner, 'route': event.soc_route,
    'artifact_matches': event.artifact_matches,
    'destination_approved': event.destination_approved,
}

### Time semantics

`occurred_at` is source event time, `observed_at` is when the producer observed it, and the argument to `normalize` is trusted ingest time. Production rules must define clock-skew, delay, late-event, and watermark behavior rather than silently collapsing these clocks.

In [ ]:
otel = event.otel_log_projection()
ocsf = event.ocsf_projection()
{
    'otel_event': otel['EventName'],
    'otel_trace': otel['TraceId'],
    'otel_resource': otel['Resource'],
    'ocsf_version_policy': ocsf['metadata']['version'],
    'ocsf_unmapped': sorted(ocsf['unmapped']),
}

### Mapping is not proof

The OTel projection uses stable log-model concepts. The OCSF projection explicitly says `deployment-selected`: production must pin and validate a release, choose a class, preserve unmapped MCP evidence, and regression-test upgrades. ECS requires the same discipline.

In [ ]:
tampered = replace(signed, event=replace(signed.event, destination='tickets.example.com'))
try:
    normalizer.normalize(tampered, NOW + timedelta(seconds=2))
except module['SOCError'] as exc:
    tamper_result = exc.code
tamper_result

In [ ]:
sensitive = module['event_fixture'](emitter, event_id='EV-SECRET', attributes=(('access_token', 'do-not-log'),))
try:
    normalizer.normalize(sensitive, NOW + timedelta(seconds=2))
except module['SOCError'] as exc:
    privacy_result = exc.code
privacy_result

### Reflection

Why is source-side minimization stronger than redacting a token after it has passed through collectors, queues, indexes, cases, and analyst exports? Which stable pseudonymous reference would still support your investigation?

## 3. Admit evidence with explicit replay behavior

The store partitions by authenticated tenant, bounds state, treats the same event ID and digest as an idempotent duplicate, and rejects the same ID with different signed content.

In [ ]:
first_ingest = store.ingest(event)
second_ingest = store.ingest(event)
(first_ingest.status, second_ingest.status, len(store.partitions[event.tenant_id]))

## 4. Detect and preserve evidence provenance

Rules emit candidate alerts. They do not prove maliciousness and cannot authorize response. Each candidate binds a rule version, exact tenant/entity, admitted event IDs, evidence digest, reason code, playbook, and detection time.

In [ ]:
alerts = detector.evaluate(event, NOW + timedelta(seconds=3))
[(a.rule_id, a.rule_version, a.severity, a.evidence_event_ids, a.reason_codes) for a in alerts]

In [ ]:
alert = next(a for a in alerts if a.rule_id == 'MCP-SOC-003')
assert all(store.contains(alert.tenant_id, evidence_id) for evidence_id in alert.evidence_event_ids)
{'fingerprint': alert.fingerprint, 'evidence_digest': alert.evidence_digest, 'playbook': alert.playbook}

### Multi-event correlation

The lab also detects a policy denial followed by a successful effect for the same tenant, server, and operation; three recent denials by the same pseudonymous principal/server; and a high-risk allowed call that lacks a resource outcome after a grace period. Exact keys prevent global or cross-tenant joins.

In [ ]:
e2, n2, s2, d2, _, _, _ = module['demo_components']()
denied = n2.normalize(module['event_fixture'](e2, event_id='EV-DENY', kind=module['EventKind'].POLICY, policy_decision='deny', outcome=module['Outcome'].DENIED), NOW + timedelta(seconds=2))
effect = n2.normalize(module['event_fixture'](e2, event_id='EV-EFFECT', sequence=2, occurred_at=NOW + timedelta(seconds=2), observed_at=NOW + timedelta(seconds=3), kind=module['EventKind'].RESOURCE, outcome=module['Outcome'].SUCCEEDED), NOW + timedelta(seconds=4))
s2.ingest(denied); s2.ingest(effect)
[(a.rule_id, a.evidence_event_ids) for a in d2.evaluate(effect, NOW + timedelta(seconds=5)) if a.rule_id == 'MCP-SOC-005']

### Sigma and backend validation

A Sigma source rule is reviewable and portable, but its generated query still depends on schema mappings and backend semantics. Pin the taxonomy/backend, inspect compiled queries, and run labelled malicious, benign, null, duplicate, late, and boundary fixtures in every target SIEM.

## 5. Route and create a case idempotently

Tenant routing comes from trusted configuration. Stable alert identity handles unknown-result retries. Only exact, bounded non-critical duplicates are suppressed; critical alerts bypass fingerprint suppression.

In [ ]:
exporter = module['InMemorySpanExporter']()
provider = module['TracerProvider'](resource=module['Resource'].create({'service.name': 'course-27-notebook'}))
provider.add_span_processor(module['SimpleSpanProcessor'](exporter))
case = module['instrumented_route'](router, alert, NOW + timedelta(seconds=4), provider.get_tracer('course-27'))
same_case = router.route(alert, NOW + timedelta(seconds=5))
provider.force_flush()
(case.case_id == same_case.case_id, exporter.get_finished_spans()[0].name, case.route)

In [ ]:
case = router.transition(case.case_id, module['CaseState'].OPEN, module['CaseState'].ACKNOWLEDGED)
case = router.transition(case.case_id, module['CaseState'].ACKNOWLEDGED, module['CaseState'].RESPONSE_PENDING)
case.state

## 6. Authorize—not infer—response

An alert or analyst creates a proposal. A trusted approval service binds the exact case, tenant, action, target, reason, state version, requester, purpose, and expiry. The executor rechecks target scope and state, and consumes the approval once.

In [ ]:
proposal = module['ResponseProposal'](
    'PROP-NB', case.case_id, case.tenant_id, module['ActionKind'].DISABLE_SERVER,
    'server:support', 'unapproved egress with admitted signed evidence',
    7, 'analyst:oncall', NOW + timedelta(seconds=5),
)
receipt = approvals.issue(proposal, 'commander:oncall', 'incident_commander', NOW + timedelta(seconds=6))
(receipt.purpose, receipt.proposal_digest == module['digest'](proposal.payload()), receipt.expires_at)

In [ ]:
execution = controller.execute('EXEC-NB', proposal, receipt, NOW + timedelta(seconds=7))
verification = module['IndependentVerifier']().verify_disabled(controller, execution.operation_id, execution.target_id, NOW + timedelta(seconds=8))
assert verification.verified
case = router.transition(case.case_id, module['CaseState'].RESPONSE_PENDING, module['CaseState'].CONTAINED)
(execution.status, verification.observed_enabled, verification.verified, case.state)

### Reflection

Why is an accepted containment command insufficient to close the case? Which independent boundary can verify that an old credential, server binding, workload, or destination can no longer produce the forbidden effect?

## 7. Measure with labelled populations

Alert count is not coverage. Use malicious and valid episodes with named label sources, time windows, and denominators. Preserve `None` where no denominator exists instead of reporting a fabricated perfect score.

In [ ]:
Episode = module['LabelledEpisode']
episodes = [
    Episode('M1', True, 0, NOW, NOW + timedelta(seconds=10), True, True),
    Episode('M2', True, 0, NOW, None, False, False),
    Episode('V1', False, 500, NOW, NOW + timedelta(seconds=2), True, False),
    Episode('V2', False, 500, NOW, None, False, False),
]
module['detection_metrics'](episodes)

## Production extension

Replace the in-memory store/router/controller with governed adapters: authenticated OTLP and a hardened Collector; a pinned OCSF or ECS mapping; versioned Sigma content compiled and tested in the target SIEM; idempotent case and SOAR APIs; a real approval authority; and independent control/resource reads. Preserve the same typed contracts and tests at every adapter seam.

## Exercises and final Reflection

1. Add an out-of-order policy/effect fixture and define watermark behavior.
2. Map the event to one pinned OCSF or ECS release and document every loss.
3. Write MCP-SOC-003 in Sigma, compile it for two SIEMs, and compare labelled results.
4. Add reconciliation for a timed-out case creation.
5. Threat-model producer-key theft, telemetry loss, cross-tenant joins, alert flooding, suppression abuse, approval replay, and forged outcome evidence.

Reflection: at which exact line does observation become proposal, proposal become authorized action, and action become independently verified outcome?